In [3]:
import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 3.0.6


# Olist E-Commerce Data Exploration

## Project Objective

Analyze e-commerce sales, customer behavior, product performance,
delivery operations, and customer satisfaction to identify
actionable business insights.

### Tools
- Python
- Pandas
- PostgreSQL
- SQL
- Power BI

### Dataset
Olist Brazilian E-Commerce Public Dataset

In [4]:
from pathlib import Path
import pandas as pd

# Location of our raw data
data_path = Path("../data/raw")

# Load all Olist datasets
customers = pd.read_csv(data_path / "olist_customers_dataset.csv")
geolocation = pd.read_csv(data_path / "olist_geolocation_dataset.csv")
orders = pd.read_csv(data_path / "olist_orders_dataset.csv")
order_items = pd.read_csv(data_path / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(data_path / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(data_path / "olist_order_reviews_dataset.csv")
products = pd.read_csv(data_path / "olist_products_dataset.csv")
sellers = pd.read_csv(data_path / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    data_path / "product_category_name_translation.csv"
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [5]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"{name:25} {df.shape}")

customers                 (99441, 5)
geolocation               (1000163, 5)
orders                    (99441, 8)
order_items               (112650, 7)
order_payments            (103886, 5)
order_reviews             (99224, 7)
products                  (32951, 9)
sellers                   (3095, 4)
category_translation      (71, 2)


In [9]:
orders.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [10]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [11]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [12]:
orders["order_status"].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [13]:
pd.crosstab(
    orders["order_status"],
    orders["order_delivered_customer_date"].isna()
)

order_delivered_customer_date,False,True
order_status,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


In [14]:
pd.crosstab(
    orders["order_status"],
    orders["order_approved_at"].isna()
)

order_approved_at,False,True
order_status,,
approved,2,0
canceled,484,141
created,0,5
delivered,96464,14
invoiced,314,0
processing,301,0
shipped,1107,0
unavailable,609,0


In [15]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [16]:
orders["customer_id"].nunique()

99441

In [18]:
pd.crosstab(
    orders["order_status"],
    orders["order_delivered_carrier_date"].isna()
)

order_delivered_carrier_date,False,True
order_status,,
approved,0,2
canceled,75,550
created,0,5
delivered,96476,2
invoiced,0,314
processing,0,301
shipped,1107,0
unavailable,0,609


In [19]:
# Store all 9 datasets in one dictionary
tables = {
    "customers": customers,
    "geolocation": geolocation,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

print("Number of tables:", len(tables))
print("Table names:", list(tables.keys()))

Number of tables: 9
Table names: ['customers', 'geolocation', 'orders', 'order_items', 'order_payments', 'order_reviews', 'products', 'sellers', 'category_translation']


In [20]:
# Create a high-level profile for all tables

profile = []

for table_name, df in tables.items():
    profile.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "total_nulls": df.isna().sum().sum()
    })

profile_df = pd.DataFrame(profile)

profile_df

,table,rows,columns,duplicate_rows,total_nulls
0,customers,99441,5,0,0
1,geolocation,1000163,5,261831,0
2,orders,99441,8,0,4908
3,order_items,112650,7,0,0
4,order_payments,103886,5,0,0
5,order_reviews,99224,7,0,145903
6,products,32951,9,0,2448
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


In [21]:
# Create a column-level data quality profile

column_profile = []

for table_name, df in tables.items():

    for column in df.columns:

        column_profile.append({
            "table": table_name,
            "column": column,
            "dtype": str(df[column].dtype),
            "null_count": df[column].isna().sum(),
            "null_percentage": round(
                df[column].isna().mean() * 100, 2
            ),
            "unique_values": df[column].nunique(dropna=True)
        })

column_profile_df = pd.DataFrame(column_profile)

column_profile_df

,table,column,dtype,null_count,null_percentage,unique_values
0,customers,customer_id,str,0,0.00,99441
1,customers,customer_unique_id,str,0,0.00,96096
2,customers,customer_zip_code_prefix,int64,0,0.00,14994
3,customers,customer_city,str,0,0.00,4119
4,customers,customer_state,str,0,0.00,27
5,geolocation,geolocation_zip_code_prefix,int64,0,0.00,19015
6,geolocation,geolocation_lat,float64,0,0.00,717360
7,geolocation,geolocation_lng,float64,0,0.00,717613
8,geolocation,geolocation_city,str,0,0.00,8011
9,geolocation,geolocation_state,str,0,0.00,27


In [22]:
# Investigate missing values in the order_reviews table

review_null_summary = order_reviews.isna().sum()

review_null_summary = review_null_summary[
    review_null_summary > 0
]

review_null_summary

review_comment_title      87656
review_comment_message    58247
dtype: int64

In [23]:
# Compare missing comments with review scores

review_comment_analysis = (
    order_reviews
    .groupby("review_score")
    .agg(
        total_reviews=("review_id", "count"),
        missing_titles=("review_comment_title", lambda x: x.isna().sum()),
        missing_messages=("review_comment_message", lambda x: x.isna().sum())
    )
)

review_comment_analysis["missing_title_pct"] = (
    review_comment_analysis["missing_titles"]
    / review_comment_analysis["total_reviews"]
    * 100
).round(2)

review_comment_analysis["missing_message_pct"] = (
    review_comment_analysis["missing_messages"]
    / review_comment_analysis["total_reviews"]
    * 100
).round(2)

review_comment_analysis

,total_reviews,missing_titles,missing_messages,missing_title_pct,missing_message_pct
review_score,,,,,
1,11424,9551,2679,83.60,23.45
2,3151,2673,1006,84.83,31.93
3,8179,7355,4622,89.93,56.51
4,19142,17407,13166,90.94,68.78
5,57328,50670,36774,88.39,64.15


In [24]:
# Check whether review comments are missing together

review_comment_presence = pd.DataFrame({
    "title_missing": order_reviews["review_comment_title"].isna(),
    "message_missing": order_reviews["review_comment_message"].isna()
})

review_comment_presence.value_counts()

title_missing  message_missing
True           True               56518
               False              31138
False          False               9839
               True                1729
Name: count, dtype: int64

In [26]:
# Investigate the pattern of missing values in products

product_missing_cols = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

missing_all_four = products[
    product_missing_cols
].isna().all(axis=1).sum()

print(
    "Rows missing all four fields:",
    missing_all_four
)

print("\nMissing values by column:")
print(
    products[product_missing_cols]
    .isna()
    .sum()
)

Rows missing all four fields: 610

Missing values by column:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64
